# First-7-Days Prospective Evaluation

**Goal**: Apply the last-7-days trained models to first-7-days patient data.

**Hypothesis**: At the start of monitoring (first 7 days), patients have not yet
deteriorated — so all predictions should ideally be **label 0**.
High label-0 accuracy here means the model is not over-triggering early alarms.

**Setup**:
- Models trained on: last-7-days data (last 7 days before hospital admission)
- Data evaluated on: first-7-days embeddings (first 7 days of monitoring)
- 6 models × 3 dims (32 / 64 / 128) = **18 evaluations**
- No re-training. No confusion matrix. Accuracy table only.

**Expected result**: label-0 accuracy high, label-1 accuracy low
(model should NOT fire on early-stage patients)


---
## Step 1 — Install & Import


In [1]:
!pip install -q xgboost scikit-learn joblib


In [2]:
import os
import json as _json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import accuracy_score, roc_auc_score, f1_score

print('All imports OK')


All imports OK


---
## Step 2 — Mount Drive & Set Paths

Points to:
- First-7-days embeddings (produced by the first-7-days transformer notebook)
- Last-7-days trained models (saved in `5models/last7days/`)


In [3]:
from google.colab import drive
drive.mount('/content/drive')

BASE_MODEL_DIR = "/content/drive/My Drive/Monmouth_University/2025_RA/Medical_AI/Best_Model"

# First-7-days transformer embedding folders (one per dim)
FIRST7_EMB_ROOT = BASE_MODEL_DIR   # embeddings inside daily_embedding_classifier_first7days_{N}dim/

# Last-7-days trained classifier models
MODELS_ROOT = os.path.join(BASE_MODEL_DIR, 'classifier_5models', 'last7days_notime')

EMBED_DIMS  = [32, 64, 128]
ALL_DAYS    = list(range(7))   # 0..6

MODEL_NAMES = [
    'xgboost',
    'logistic_regression',
    'random_forest',
    'svm',
    'decision_tree',
    'pca_xgboost',
]
MODEL_LABELS = {
    'xgboost':             'XGBoost',
    'logistic_regression': 'Logistic Regression',
    'random_forest':       'Random Forest',
    'svm':                 'SVM',
    'decision_tree':       'Decision Tree',
    'pca_xgboost':         'PCA + XGBoost',
}

print(f'Base model dir : {BASE_MODEL_DIR}')
print(f'Models root    : {MODELS_ROOT}')
print(f'Dims           : {EMBED_DIMS}')
print(f'Models         : {MODEL_NAMES}')

# Verify model files exist
print('\nVerifying model files:')
missing = []
for dim in EMBED_DIMS:
    for mname in MODEL_NAMES:
        path = os.path.join(MODELS_ROOT, f'{dim}dim', mname, 'model.joblib')
        status = '✅' if os.path.exists(path) else '❌ MISSING'
        if not os.path.exists(path):
            missing.append(path)
        print(f'  {status}  {dim}dim/{mname}/model.joblib')
if missing:
    print(f'\n⚠️  {len(missing)} model files missing — run Step 5 of the classifier notebook first')
else:
    print('\n✅ All 18 model files present')


Mounted at /content/drive
Base model dir : /content/drive/My Drive/Monmouth_University/2025_RA/Medical_AI/Best_Model
Models root    : /content/drive/My Drive/Monmouth_University/2025_RA/Medical_AI/Best_Model/classifier_5models/last7days_notime
Dims           : [32, 64, 128]
Models         : ['xgboost', 'logistic_regression', 'random_forest', 'svm', 'decision_tree', 'pca_xgboost']

Verifying model files:
  ✅  32dim/xgboost/model.joblib
  ✅  32dim/logistic_regression/model.joblib
  ✅  32dim/random_forest/model.joblib
  ✅  32dim/svm/model.joblib
  ✅  32dim/decision_tree/model.joblib
  ✅  32dim/pca_xgboost/model.joblib
  ✅  64dim/xgboost/model.joblib
  ✅  64dim/logistic_regression/model.joblib
  ✅  64dim/random_forest/model.joblib
  ✅  64dim/svm/model.joblib
  ✅  64dim/decision_tree/model.joblib
  ✅  64dim/pca_xgboost/model.joblib
  ✅  128dim/xgboost/model.joblib
  ✅  128dim/logistic_regression/model.joblib
  ✅  128dim/random_forest/model.joblib
  ✅  128dim/svm/model.joblib
  ✅  128dim/dec

---
## Step 3 — Load First-7-Days Embeddings & Build Patient Features

Same `build_patient_features` logic as the classifier notebook:
concatenate 7 daily embeddings per patient → one feature vector per patient.
Missing days → zero-padded.


In [4]:
def build_patient_features(emb_df, emb_cols, emb_dim):
    """
    One row per patient = concatenated embeddings for days 0..6.
    Missing days → zero-padded.
    Returns X [n_patients, 7*emb_dim], y [n_patients], patient_ids.
    """
    patient_ids = emb_df['patient_id'].unique()
    X_rows, y_rows, pids_out = [], [], []
    for pid in patient_ids:
        sub   = emb_df[emb_df['patient_id'] == pid]
        label = int(sub['label'].iloc[0])
        day_vecs = []
        for d in ALL_DAYS:
            rows = sub[sub['unified_day'] == d]
            if len(rows) > 0:
                vec = rows[emb_cols].values.mean(axis=0).astype(np.float32)
            else:
                vec = np.zeros(emb_dim, dtype=np.float32)
            day_vecs.append(vec)
        X_rows.append(np.concatenate(day_vecs))
        y_rows.append(label)
        pids_out.append(pid)
    return np.array(X_rows), np.array(y_rows), np.array(pids_out)


# Load first-7-days embeddings for each dim
first7_data = {}   # {dim: {'X': ..., 'y': ..., 'pids': ...}}

for EMBED_DIM in EMBED_DIMS:
    emb_folder = os.path.join(BASE_MODEL_DIR,
                     f'daily_embedding_classifier_first7days_{EMBED_DIM}dim_notime')

    # Combine all splits — we evaluate the full dataset (no train/val/test split needed)
    splits = []
    for split_name in ['train', 'val', 'test']:
        path = os.path.join(emb_folder, f'emb_{split_name}.parquet')
        if os.path.exists(path):
            splits.append(pd.read_parquet(path))
        else:
            print(f'  ⚠️  Missing: {path}')
    if not splits:
        print(f'  ❌  No embeddings found for dim={EMBED_DIM}')
        continue

    emb_all  = pd.concat(splits, ignore_index=True)
    emb_cols = [c for c in emb_all.columns if c.startswith('emb_')]

    X, y, pids = build_patient_features(emb_all, emb_cols, EMBED_DIM)
    first7_data[EMBED_DIM] = {'X': X, 'y': y, 'pids': pids, 'emb_all': emb_all}

    n_label0 = (y == 0).sum()
    n_label1 = (y == 1).sum()
    print(f'  dim={EMBED_DIM}: {X.shape}  '
          f'label0={n_label0}  label1={n_label1}  '
          f'feature_vec={EMBED_DIM*7}D')

print('\n✅ First-7-days features built')


  dim=32: (85, 224)  label0=57  label1=28  feature_vec=224D
  dim=64: (85, 448)  label0=57  label1=28  feature_vec=448D
  dim=128: (85, 896)  label0=57  label1=28  feature_vec=896D

✅ First-7-days features built


---
## Step 4 — Load Models & Evaluate

For each dim × model:
1. Load `model.joblib` from `5models/last7days/{dim}dim/{model}/`
2. Run predictions on first-7-days feature vectors
3. Report accuracy — overall, label-0, label-1

**Note on label-1 patients**: some patients in the first-7-days data are label 1
(they eventually went to hospital). The model was trained on their *last* 7 days.
Here we score their *first* 7 days — we expect the model to predict 0 (not yet sick).
Low label-1 accuracy = good (model not firing early).
High label-1 accuracy here would mean over-sensitivity.


In [5]:
all_results = []
OUT_DIR = os.path.join(MODELS_ROOT, 'first7days_eval')
os.makedirs(OUT_DIR, exist_ok=True)

SELECTED_MODELS = ['xgboost', 'logistic_regression', 'random_forest',
               'svm', 'decision_tree', 'pca_xgboost']

for EMBED_DIM in EMBED_DIMS:
    if EMBED_DIM not in first7_data:
        print(f'⚠️  dim={EMBED_DIM} not loaded — skipping')
        continue

    X    = first7_data[EMBED_DIM]['X']
    y    = first7_data[EMBED_DIM]['y']
    pids = first7_data[EMBED_DIM]['pids']

    print('\n' + '█'*65)
    print(f'  dim={EMBED_DIM}  |  {X.shape[0]} patients  |  '
          f'true_label0={( y==0).sum()}  true_label1={(y==1).sum()}')
    print('█'*65)

    for mname in SELECTED_MODELS:
        model_path = os.path.join(MODELS_ROOT, f'{EMBED_DIM}dim', mname, 'model.joblib')
        label      = MODEL_LABELS[mname]

        if not os.path.exists(model_path):
            print(f'  ❌  {label}: not found — skipping')
            continue

        clf   = joblib.load(model_path)
        try:    probs = clf.predict_proba(X)[:, 1]
        except: probs = clf.decision_function(X)
        preds = (probs >= 0.5).astype(int)

        acc_L0  = accuracy_score(y[y==0], preds[y==0]) if (y==0).sum() > 0 else float('nan')
        n_pred0 = (preds == 0).sum()
        n_pred1 = (preds == 1).sum()

        print(f'  {label:<20}  AccL0={acc_L0:.3f}  '
              f'pred_label0={n_pred0}  pred_label1={n_pred1}')

        all_results.append({
            'embed_dim': EMBED_DIM,
            'model':     label,
            'acc_L0':    round(acc_L0, 4),
            'n_pred0':   int(n_pred0),
            'n_pred1':   int(n_pred1),
        })

print('\n✅ Done')


█████████████████████████████████████████████████████████████████
  dim=32  |  85 patients  |  true_label0=57  true_label1=28
█████████████████████████████████████████████████████████████████
  XGBoost               AccL0=0.000  pred_label0=0  pred_label1=85
  Logistic Regression   AccL0=0.035  pred_label0=5  pred_label1=80
  Random Forest         AccL0=0.070  pred_label0=5  pred_label1=80
  SVM                   AccL0=1.000  pred_label0=85  pred_label1=0
  Decision Tree         AccL0=1.000  pred_label0=85  pred_label1=0
  PCA + XGBoost         AccL0=0.000  pred_label0=0  pred_label1=85

█████████████████████████████████████████████████████████████████
  dim=64  |  85 patients  |  true_label0=57  true_label1=28
█████████████████████████████████████████████████████████████████
  XGBoost               AccL0=0.000  pred_label0=0  pred_label1=85
  Logistic Regression   AccL0=0.000  pred_label0=0  pred_label1=85
  Random Forest         AccL0=0.877  pred_label0=71  pred_label1=14
  SVM     